# Cicero Digital – Netzwerke mit pathpyG

Dieses Notebook ist als **Seminarversion** gedacht: einfach, schrittweise und mit Platz zum eigenen Experimentieren.

Wir arbeiten hier bewusst in **zwei Schritten**:

1. Wir bauen **ein kleines Netzwerk von Hand**.
2. Danach erstellen wir **ein PROPN-Kookkurrenznetzwerk** aus den CoNLL-Dateien.

Wichtig ist dabei nicht nur das Resultat, sondern auch die Frage:
**Wie entstehen Netzwerke eigentlich aus Daten und Modellierungsentscheidungen?**


## 0. Setup

Falls noetig, die Installation unten aktivieren.

Fuer dieses Notebook verwenden wir bewusst **pathpyG** fuer die Netzwerke und deren Visualisierung.


In [ ]:
# Falls noetig:
# !pip install torch
# !pip install torch_geometric
# !pip install git+https://github.com/pathpy/pathpyG.git


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from itertools import combinations

import pandas as pd

try:
    import pathpyG as pp
    HAS_PATHPYG = True
    print("pathpyG wurde erfolgreich geladen.")
except Exception as e:
    HAS_PATHPYG = False
    print("pathpyG konnte nicht geladen werden.")
    print(e)


In [ ]:
def safe_plot(graph, **kwargs):
    """Kleine Hilfsfunktion fuer pathpyG-Plots."""
    if not HAS_PATHPYG:
        print("pathpyG fehlt. Bitte das Notebook in einer Umgebung mit pathpyG ausfuehren.")
        return None
    try:
        return pp.plot(graph, **kwargs)
    except Exception as e:
        print("Plot konnte in dieser Umgebung nicht erzeugt werden.")
        print(e)
        return None


def density_from_counts(n, m, directed=False):
    if n <= 1:
        return 0.0
    if directed:
        return m / (n * (n - 1))
    return m / (n * (n - 1) / 2)


def weighted_degree_from_edge_table(edge_df, directed=False):
    """Einfacher gewichteter Degree direkt aus einer Kantentabelle."""
    nodes = sorted(set(edge_df['source']).union(set(edge_df['target'])))
    degree = {n: set() for n in nodes}
    weighted_degree = {n: 0 for n in nodes}

    for _, row in edge_df.iterrows():
        s, t, w = row['source'], row['target'], row['weight']
        degree[s].add(t)
        weighted_degree[s] += w
        if not directed:
            degree[t].add(s)
            weighted_degree[t] += w
        else:
            if t not in degree:
                degree[t] = set()
            if t not in weighted_degree:
                weighted_degree[t] = 0

    return pd.DataFrame([
        {
            'node': n,
            'degree': len(degree[n]),
            'weighted_degree': weighted_degree[n]
        }
        for n in nodes
    ])


def try_betweenness(graph):
    if not HAS_PATHPYG:
        return None
    try:
        values = pp.algorithms.centrality.betweenness_centrality(graph)
        return pd.DataFrame({
            'node': list(values.keys()),
            'betweenness': list(values.values())
        }).sort_values('betweenness', ascending=False).reset_index(drop=True)
    except Exception as e:
        print("Betweenness konnte hier nicht berechnet werden.")
        print(e)
        return None


def filter_edge_table(edge_df, top_nodes=None, min_edge_weight=1):
    out = edge_df.copy()
    out = out[out['weight'] >= min_edge_weight].copy()
    if top_nodes is not None:
        out = out[
            out['source'].isin(top_nodes) &
            out['target'].isin(top_nodes)
        ].copy()
    return out.reset_index(drop=True)


def build_graph_from_edge_table(edge_df, undirected=True):
    if not HAS_PATHPYG:
        return None
    edge_list = [tuple(x) for x in edge_df[['source', 'target']].itertuples(index=False, name=None)]
    g = pp.Graph.from_edge_list(edge_list)
    if undirected:
        try:
            g = g.to_undirected()
        except Exception:
            pass
    return g


def ego_edge_table(edge_df, center_node):
    ego = edge_df[
        (edge_df['source'] == center_node) |
        (edge_df['target'] == center_node)
    ].copy()
    return ego.sort_values(['weight', 'source', 'target'], ascending=[False, True, True]).reset_index(drop=True)


## 1. Erstes Beispiel: ein kleines Netzwerk von Hand

Wir beginnen mit einem **Mini-Netzwerk**, das wir selber definieren.

Warum?

- Wir sehen sofort, was **Knoten**, **Kanten** und **Gewichte** sind.
- Wir koennen einige **wichtige Masse** in Ruhe anschauen.
- Erst danach gehen wir zu den echten Daten.


In [ ]:
manual_edges = pd.DataFrame([
    {'source': 'Cicero',   'target': 'Atticus',   'weight': 8},
    {'source': 'Cicero',   'target': 'Brutus',    'weight': 4},
    {'source': 'Cicero',   'target': 'Caesar',    'weight': 3},
    {'source': 'Atticus',  'target': 'Brutus',    'weight': 2},
    {'source': 'Brutus',   'target': 'Cassius',   'weight': 5},
    {'source': 'Caesar',   'target': 'Pompeius',  'weight': 6},
    {'source': 'Pompeius', 'target': 'Crassus',   'weight': 4},
    {'source': 'Cicero',   'target': 'Pompeius',  'weight': 2},
])

manual_edges


### 1.1 Ein paar wichtige Masse

Wir halten es bewusst einfach.

- **Anzahl Knoten**: Wie viele Akteure gibt es?
- **Anzahl Kanten**: Wie viele Verbindungen gibt es?
- **Dichte**: Wie stark ist das Netzwerk insgesamt verbunden?
- **Degree**: Mit wie vielen anderen Knoten ist ein Knoten direkt verbunden?
- **Gewichteter Degree**: Wie stark ist ein Knoten insgesamt eingebunden?
- **Betweenness**: Welche Knoten liegen oft auf Wegen zwischen anderen?

Diese Masse reichen fuer den Einstieg voellig aus.


In [ ]:
manual_graph = build_graph_from_edge_table(manual_edges, undirected=False)

manual_nodes = sorted(set(manual_edges['source']).union(set(manual_edges['target'])))
manual_basic = pd.DataFrame([
    {'Mass': 'Knoten', 'Wert': len(manual_nodes)},
    {'Mass': 'Kanten', 'Wert': len(manual_edges)},
    {'Mass': 'Dichte', 'Wert': round(density_from_counts(len(manual_nodes), len(manual_edges), directed=False), 4)}
])

manual_basic


In [ ]:
manual_stats = weighted_degree_from_edge_table(manual_edges, directed=True).sort_values(
    ['weighted_degree', 'degree', 'node'],
    ascending=[False, False, True]
).reset_index(drop=True)

manual_stats


In [ ]:
manual_betweenness = try_betweenness(manual_graph)
manual_betweenness


### 1.2 Visualisierung mit pathpyG

Hier ist ein kleines, gut lesbares Netzwerk.

Die Knoten groesse koppeln wir grob an den **gewichteten Degree**. So sieht man sofort, welche Akteure staerker eingebunden sind.


In [ ]:
if HAS_PATHPYG:
    # Reihenfolge der Knoten und Kanten im aktuellen Plot-Graphen
    node_order = list(manual_graph.nodes)
    edge_order = list(manual_graph.edges)

    # Hilfs-Mappings aus den manuellen Daten
    node_weight_map = dict(zip(manual_stats['node'], manual_stats['weighted_degree']))
    edge_weight_map = {
        (row['source'], row['target']): row['weight']
        for _, row in manual_edges.iterrows()
    }

    # Groessen als LISTEN statt Dicts
    node_sizes = [
        max(8, min(40, node_weight_map.get(node, 1) * 1.2))
        for node in node_order
    ]

    edge_sizes = [
        max(1, min(8, edge_weight_map.get((u, v), edge_weight_map.get((v, u), 1))))
        for (u, v) in edge_order
    ]

    edge_labels = [
        str(edge_weight_map.get((u, v), edge_weight_map.get((v, u), "")))
        for (u, v) in edge_order
    ]

    safe_plot(
        manual_graph,
        layout='forceatlas2',
        node_size=node_sizes,
        edge_size=edge_sizes,
        edge_label=edge_labels,
        show_labels=True,
    )

### Platz fuer eigenes Experimentieren

Probiere nun selbst etwas aus:

- Fuege eine neue Kante hinzu.
- Aendere ein Gewicht.
- Fuege einen neuen Akteur ein.
- Beobachte danach, was mit Degree, gewichteter Einbindung und Betweenness passiert.


In [ ]:
# Beispiel zum Weiterarbeiten:
# manual_edges_2 = manual_edges.copy()
# manual_edges_2.loc[len(manual_edges_2)] = ['Atticus', 'Caesar', 3]
# 
# manual_graph_2 = build_graph_from_edge_table(manual_edges_2, undirected=True)
# manual_stats_2 = weighted_degree_from_edge_table(manual_edges_2, directed=False)
# manual_stats_2.sort_values('weighted_degree', ascending=False)


## 2. Zweites Beispiel: ein PROPN-Kookkurrenznetzwerk

Jetzt wechseln wir zu den CoNLL-Dateien im `outputs`-Ordner.

**Regel:**
Zwischen zwei Eigennamen entsteht **genau dann** eine Kante, wenn beide **im selben Brief** vorkommen.

Das Gewicht einer Kante ist also die **Anzahl der Briefe**, in denen dieses Paar gemeinsam vorkommt.

> **Methodische Bemerkung:** Mit dem POS-Tag `PROPN` allein koennen wir **Personen und Orte nicht sauber unterscheiden**. Ein solches Netzwerk ist deshalb zunaechst ein **Eigennamen-Netzwerk**, nicht automatisch ein reines Personennetzwerk.


In [ ]:
base_dir = Path('outputs')
candidate_dirs = [
    base_dir / 'conllu_letters'
]

conll_files = []
for folder in candidate_dirs:
    if folder.exists():
        conll_files.extend(sorted(folder.glob('*.conll')))
        conll_files.extend(sorted(folder.glob('*.conllu')))

seen = set()
conll_files = [p for p in conll_files if not (str(p) in seen or seen.add(str(p)))]

print('Gefundene CoNLL-Dateien:', len(conll_files))
conll_files[:5]


In [ ]:
def read_propns_from_conll(path):
    """Liest eine CoNLL/CoNLL-U-Datei und extrahiert alle PROPN-Tokens eines Briefs."""
    propns = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line or line.startswith("#"):
                continue

            parts = line.split("\t")
            if len(parts) < 4:
                continue

            token_id = parts[0]

            # Multiword tokens oder empty nodes überspringen
            if "-" in token_id or "." in token_id:
                continue

            form = parts[1].strip()
            lemma = parts[2].strip() if len(parts) > 2 else ""
            upos = parts[3].strip() if len(parts) > 3 else ""

            if upos != "PROPN":
                continue

            label = lemma if lemma not in {"", "_"} else form
            label = label.strip()

            if label:
                propns.append(label)

    return propns

In [ ]:
doc_propns = []

for path in conll_files:
    propns = read_propns_from_conll(path)

    # Wichtig: Fuer das Netzwerk zaehlt nur, ob ein Name in einem Brief vorkommt,
    # nicht wie oft er innerhalb desselben Briefs vorkommt.
    unique_propns = sorted(set(propns))

    doc_propns.append({
        "doc_id": path.stem,
        "n_propns_total": len(propns),
        "n_propns_unique": len(unique_propns),
        "propns": unique_propns
    })

doc_propns_df = pd.DataFrame(doc_propns)

print("Briefe mit mindestens einem PROPN:", (doc_propns_df["n_propns_unique"] > 0).sum())
doc_propns_df.head()


In [ ]:
propn_edge_counter = Counter()

for _, row in doc_propns_df.iterrows():
    names = row["propns"]

    # Eine Kante entsteht nur dann, wenn zwei verschiedene Eigennamen
    # im gleichen Brief gemeinsam vorkommen.
    if len(names) < 2:
        continue

    for a, b in combinations(sorted(names), 2):
        propn_edge_counter[(a, b)] += 1

propn_edges = pd.DataFrame(
    [
        {"source": a, "target": b, "weight": w}
        for (a, b), w in propn_edge_counter.items()
    ]
).sort_values(["weight", "source", "target"], ascending=[False, True, True])

print("Kanten im PROPN-Netzwerk:", len(propn_edges))
propn_edges.head(15)


### 2.1 Ein paar einfache Masse

Auch hier schauen wir zuerst auf die Grundstruktur.

Gerade bei Kookkurrenznetzwerken ist das wichtig: Sie werden schnell sehr dicht. Deshalb ist nicht jede Visualisierung automatisch lesbar.


In [ ]:
if len(propn_edges) > 0:
    propn_nodes = sorted(set(propn_edges['source']).union(set(propn_edges['target'])))
    propn_basic = pd.DataFrame([
        {'Mass': 'Knoten', 'Wert': len(propn_nodes)},
        {'Mass': 'Kanten', 'Wert': len(propn_edges)},
        {'Mass': 'Dichte', 'Wert': round(density_from_counts(len(propn_nodes), len(propn_edges), directed=False), 4)}
    ])
else:
    propn_basic = pd.DataFrame([
        {'Mass': 'Knoten', 'Wert': 0},
        {'Mass': 'Kanten', 'Wert': 0},
        {'Mass': 'Dichte', 'Wert': 0}
    ])

propn_basic


In [ ]:
propn_stats = weighted_degree_from_edge_table(propn_edges, directed=False).sort_values(
    ['weighted_degree', 'degree', 'node'],
    ascending=[False, False, True]
).reset_index(drop=True)

propn_stats.head(20)


### 2.2 Visualisierung: filtern statt alles auf einmal zeigen

Wenn wir **alles** plotten, entsteht schnell ein **Hairball**. Deshalb filtern wir vor der Darstellung.

Hier sind drei einfache Stellschrauben:

- `top_n_nodes`: Wie viele der staerksten Knoten wollen wir zeigen?
- `min_edge_weight`: Wie stark muss eine Kante mindestens sein?
- `show_labels`: Sollen Labels eingeblendet werden?

Das ist keine Manipulation, sondern ein ganz normaler Teil der explorativen Netzwerkanalyse.


In [ ]:
top_n_nodes = 18
min_edge_weight = 2
show_labels = True

top_nodes = propn_stats.head(top_n_nodes)['node'].tolist() if len(propn_stats) else []
propn_edges_plot = filter_edge_table(
    propn_edges,
    top_nodes=top_nodes,
    min_edge_weight=min_edge_weight
)

propn_edges_plot.head(20)


In [ ]:
# Graph DIREKT aus der Plot-Tabelle bauen
propn_edge_list = list(
    propn_edges_plot[['source', 'target']].itertuples(index=False, name=None)
)

propn_graph_plot = pp.Graph.from_edge_list(propn_edge_list)

if HAS_PATHPYG:
    # Mappings genau aus den Daten, die auch geplottet werden
    # Knotengewichte NUR aus dem geplotteten Netzwerk berechnen
    propn_node_weight_map = defaultdict(int)

    for _, row in propn_edges_plot.iterrows():
        propn_node_weight_map[row['source']] += row['weight']
        propn_node_weight_map[row['target']] += row['weight']

    
    propn_edge_weight_map = {
        (row['source'], row['target']): row['weight']
        for _, row in propn_edges_plot.iterrows()
    }

    safe_plot(
        propn_graph_plot,
        layout='forceatlas2',
        node_size={
            node: max(6, min(30, propn_node_weight_map.get(node, 1) * 0.05))
            for node in propn_graph_plot.nodes
        },
        edge_size={
            (u, v): max(1, min(10, propn_edge_weight_map.get((u, v), 1) * 0.15))
            for (u, v) in propn_graph_plot.edges
        },
        show_labels=True,
        #width=1400,
        #height=900
    )

### 2.3 Lesbarer Spezialfall: Ego-Netzwerk eines Knotens

Noch lesbarer wird es, wenn wir **nicht das ganze Netzwerk**, sondern nur die direkte Umgebung eines Knotens zeigen.

Das nennt man oft ein **Ego-Netzwerk**.


In [ ]:
center_node = propn_stats.iloc[0]['node'] if len(propn_stats) else None
center_node

In [ ]:
if center_node is not None:
    propn_ego_edges = ego_edge_table(propn_edges, center_node)
    propn_ego_edges.head(20)
else:
    print("Kein Zentrumsknoten vorhanden.")


In [ ]:
propn_ego_graph = build_graph_from_edge_table(propn_ego_edges, undirected=True) if center_node is not None else None

if HAS_PATHPYG and propn_ego_graph is not None and center_node is not None and len(propn_ego_edges) > 0:
    ego_nodes = sorted(set(propn_ego_edges['source']).union(set(propn_ego_edges['target'])))
    ego_sizes = {}
    for node in ego_nodes:
        if node == center_node:
            ego_sizes[node] = 28
        else:
            ego_sizes[node] = 14

    safe_plot(
        propn_ego_graph,
        layout='forceatlas2',
        node_size=ego_sizes,
        show_labels=True
    )
else:
    print("Ego-Netzwerk konnte nicht geplottet werden.")


### Platz fuer eigenes Experimentieren

Moegliche Dinge zum Testen:

- Setze `top_n_nodes` auf `10`, `20` oder `30`.
- Setze `min_edge_weight` auf `1`, `2` oder `3`.
- Waehle fuer das Ego-Netzwerk einen anderen `center_node`.
- Vergleiche: Was sieht man im Gesamtplot, was im Ego-Netzwerk?
- Notiere, wo die Grenze des `PROPN`-Ansatzes liegt.


In [ ]:
# Eigener Code hier


## 3. Optional: dieselben Daten als zeitliches Netzwerk denken

`pathpyG` kann auch **zeitliche Netzwerke** darstellen.

Fuer die Briefe waere das besonders spannend, wenn wir die Datierung sicher genug haben. Dann koennte man nicht nur fragen,
**wer mit wem verbunden ist**, sondern auch,
**wann** bestimmte Verbindungen sichtbar werden.


In [ ]:
if HAS_PATHPYG:
    temporal_edges_toy = [
        ('Cicero', 'Atticus', 1),
        ('Cicero', 'Brutus', 2),
        ('Brutus', 'Cassius', 3),
        ('Cicero', 'Pompeius', 4),
        ('Pompeius', 'Crassus', 5),
        ('Cicero', 'Atticus', 5),
        ('Cicero', 'Atticus', 5),
        ('Cicero', 'Pompeius', 5)
    ]

    try:
        T_toy = pp.TemporalGraph.from_edge_list(temporal_edges_toy)
        safe_plot(T_toy)
    except Exception as e:
        print("Temporales Beispiel konnte hier nicht geplottet werden.")
        print(e)


## 4. Optional: dieselben Daten als zeitliches Netzwerk denken -- jetzt du

Erstelle das temporale Netzwerk für die Cicero-Briefe mit Datierung. Bisher haben wir nur gefragt, wer mit wem in denselben Briefen vorkommt.
Nun kommt eine zusätzliche Dimension dazu: Zeit.

Nicht alle Briefe im Korpus sind gleich gut datiert. Für ein temporales Netzwerk dürfen wir deshalb nur diejenigen Briefe verwenden, für die in den Metadaten eine Datierung vorliegt.

**Aufgabe**

Erstelle ein temporales `PROPN`-Kookkurrenznetzwerk auf Basis der datierten Cicero-Briefe.

Gehe dabei in folgenden Schritten vor (bei den ersten ist schon Unterstützung da):

- Lade die Metadaten mit den Datierungen.
- Behalte nur Briefe, für die eine Datierung vorhanden ist.
- Verbinde diese Metadaten mit den CoNLL-Dateien im outputs-Ordner.
- Extrahiere pro datiertem Brief die PROPN-Token.
- Erzeuge für jeden Brief alle PROPN-Paare, die im selben Brief gemeinsam vorkommen.
- Ergänze zu jeder Kante einen Zeitstempel (z. B. Jahr oder genaueres Datum, falls vorhanden).
- Baue daraus ein temporales Netzwerk.

*Hinweise*

Für diese Aufgabe reicht zunächst oft schon das Jahr als Zeitinformation.
Falls die Datierungen unterschiedlich genau sind, darfst du das vereinfachen und nur mit Briefen arbeiten, bei denen ein Jahr vorhanden ist.
Auch hier gilt: Mit `PROPN` allein kann man Personen und Orte nicht sicher unterscheiden. Das ist für die Übung in Ordnung, sollte aber als methodische Grenze notiert werden.

**Fragen zur Auswertung**
- Welche Namen treten früh auf, welche spät?
- Verändert sich das Netzwerk über die Zeit?
- Gibt es Phasen, in denen gewisse Akteure besonders stark vernetzt sind?
- Was gewinnt man, wenn man ein statisches Netzwerk in ein temporales Netzwerk überführt?

In [ ]:
def make_doc_id(row, idx):
    corpus = str(row.get("corpus", "unknown")).strip()

    book_n = row.get("book_n")
    letter_n = row.get("letter_n")

    book_n = "x" if pd.isna(book_n) else str(book_n).strip()
    letter_n = str(idx) if pd.isna(letter_n) else str(letter_n).strip()

    return f"{corpus}_b{book_n}_l{letter_n}_r{idx}"

In [ ]:
# ============================================================
# Vorbereitung fuer ein temporales PROPN-Kookkurrenznetzwerk
# ============================================================

from pathlib import Path
from itertools import combinations
from collections import defaultdict
import pandas as pd

# 1. Metadaten laden
# Passe den Pfad ggf. an
meta_df = pd.read_csv("outputs/cicero_letters_metadata.csv")

# 2. Nur datierte Briefe behalten

# 2. doc_id genau gleich erzeugen wie oben
meta_df = meta_df.copy()
meta_df["doc_id"] = [
    make_doc_id(row, idx)
    for idx, (_, row) in enumerate(meta_df.iterrows())
]

# 3. nur datierte Briefe behalten
meta_df = meta_df.dropna(subset=["year"]).copy()
meta_df["year"] = meta_df["year"].astype(int)

# 4. Merge
merged_df = doc_propns_df.merge(
    meta_df[["doc_id", "year"]],
    on="doc_id",
    how="inner"
)

print("Matchende Briefe:", len(merged_df))
display(merged_df.head())